# 03 — Analysis and Validation

Build an `EModelAnalysisAndValidationScanConfig`, expand it via `GridScanGenerationTask`,
and run the `EModelAnalysisAndValidationTask` for each coordinate. The task seeds its
working directory from the optimisation output, merges the analysis-related
`pipeline_settings` from the blocks into `recipes.json`, then runs
`pipeline.store_optimisation_results()` → `pipeline.validation()` →
`pipeline.plot(only_validated=...)`.

**Reads from:** `obi-output/02_emodel_optimization/grid_scan/0/`.

**Writes to:** `obi-output/03_analysis_and_validation/grid_scan/0/` — the working
directory plus the new `final.json` and the `figures/L5PC/` plots (traces, scores,
distributions, currentscape if enabled).


## Imports

In [ ]:
from pathlib import Path

import obi_one as obi
from obi_one.scientific.tasks.emodel_optimization._03_analysis_and_validation.blocks import (
    AnalysisInitialize,
    AnalysisSettings,
    CurrentscapeConfig,
)


## Build the scan config

In [ ]:
previous_stage = (
    Path.cwd() / "../../../obi-output/02_emodel_optimization/grid_scan/0"
).resolve()
assert previous_stage.exists(), (
    f"{previous_stage} not found — run 02_emodel_optimization.ipynb first."
)

scan_config = obi.EModelAnalysisAndValidationScanConfig(
    initialize=AnalysisInitialize(
        previous_stage_output_path=previous_stage,
        emodel="L5PC",
        species="rat",
        brain_region="SSCX",
    ),
    analysis_settings=AnalysisSettings(
        validation_protocols=("sAHP_220",),
        plot_currentscape=True,
        save_recordings=False,
        only_validated_plots=False,
    ),
    currentscape_config=CurrentscapeConfig(figure_title="L5PC"),
)
print(scan_config.analysis_settings.to_dict(scan_config.currentscape_config))


## Run the grid scan

In [ ]:
grid_scan = obi.GridScanGenerationTask(
    form=scan_config,
    output_root="../../../obi-output/03_analysis_and_validation/grid_scan",
    coordinate_directory_option="ZERO_INDEX",
)
grid_scan.execute()

obi.run_tasks_for_generated_scan(grid_scan)


## Inspect the validated models

In [ ]:
import json

coord_root = Path(grid_scan.single_configs[0].coordinate_output_root).resolve()
print("Coordinate output:", coord_root)

final_path = coord_root / "final.json"
if final_path.exists():
    final = json.loads(final_path.read_text())
    print(f"final.json — {len(final)} model(s)")
    for emodel_name, payload in final.items():
        print(" ", emodel_name, "validated:", payload.get("validated", payload.get("passed_validation")))
else:
    print("final.json not produced yet")

figures = sorted((coord_root / "figures").rglob("*.pdf")) + sorted(
    (coord_root / "figures").rglob("*.png")
)
print(f"\nFigures: {len(figures)}")
for p in figures[:8]:
    print(" ", p.relative_to(coord_root))
